[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/saltfingers/saltfingers.ipynb)

# Salt Fingers

This example computes salt fingers, the double-diffusive convection of warm salty water lying over cold fresh water: water that is stably stratified, lighter over heavier, and still turns over, in narrow fingers. Warmth makes the upper layer light and salt makes it heavy, and with warmth enough the two layers rest one on the other. But heat diffuses through seawater about a hundred times faster than salt. A salty parcel pushed down loses its warmth sideways to the cold fresh water around it long before it loses its salt, comes out heavier than its neighbours and keeps sinking; a fresh parcel pushed up warms and keeps rising. Here the interface between the two layers grows fingers from a little noise, and a sweep over the density ratio, the mass the warmth on top takes away over the mass its salt adds, finds where a disturbance stops growing, as [Radko's lectures at the Woods Hole Geophysical Fluid Dynamics Program (2008)](https://gfd.whoi.edu/wp-content/uploads/sites/18/2018/03/lecture10_136325.pdf) set out.

The algebra is planar spacetime joined with three degenerate directions, for volume, entropy and salt, `Algebra("x+y+t-") * Algebra("v0s0n0")`. A parcel of water is modelled as a point weighed by its volume, offset along time by its mass and along the entropy's and the salt's directions by its entropy and its salt, `volume + mass * time + warmth * entropy + saltiness * salt`; a unit volume holds a little less mass the warmer it is and a little more the saltier. Salt is a second amount of the same kind as entropy: a direction with no metric, carried by the flow and spread by a drive of its own, the gradient of its chemical potential wedged with `salt`, as the temperature's gradient wedged with `entropy` spreads the entropy, and by the Gibbs–Duhem relation neither drive pushes anything. Gravity's drive pulls on the mass alone, its commutator with `volume`, `entropy` or `salt` vanishing, so heat and salt move the water only through the mass they change. The flow carries whole parcels, and the entropy and the salt it carries out of a cell are what the planes `mv.s` and `mv.n` read off one flux, `velocities & parcels`. The two amounts differ only in how fast they spread and in the sign of the mass they add, and that difference alone makes the fingers.

In the notation of the Oberbeck–Boussinesq equations for water holding heat and salt, the water obeys $\partial_t\mathbf v = -\nabla p/\rho_0 + \nu\nabla^2\mathbf v + g(\alpha T - \beta S)\,\hat{\mathbf z}$, $\nabla\cdot\mathbf v = 0$, $\partial_t T + \mathbf v\cdot\nabla T = \kappa_T\nabla^2 T$ and $\partial_t S + \mathbf v\cdot\nabla S = \kappa_S\nabla^2 S$, without the momentum's own advection here; the density ratio of evenly stratified water is $R = \alpha\,\partial_z T/(\beta\,\partial_z S)$, its fingers of wavenumber $k$ are marginal where $g\,(\beta\,\partial_z S/\kappa_S - \alpha\,\partial_z T/\kappa_T) = \nu k^4$, and they grow where $1 < R < \kappa_T/\kappa_S$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice, tee

import numpy as np

from numga import Algebra, NumpyContext
from numga.sparse import spdiag
from examples.util.animation import show_animation
from examples.fluids.saltfingers import render, scenarios
from examples.fluids.saltfingers.core import tank

# Planar spacetime, and the volume, the entropy and the salt a parcel holds: each a direction of its own, degenerate,
# since nothing measures a volume, an entropy or an amount of salt against a length or a time.
spacetime = Algebra("x+y+t-")
ga = spacetime * Algebra("v0s0n0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
axis = mv.x.dual()                                                            # [] Point-
up = mv.y.dual()                                                              # [] Point-
time = -mv.t.dual()                                                           # [] Point-
# A unit of volume, the weight of a place, a unit of entropy and a unit of salt.
volume = mv.v.dual()                                                          # [] Point-
entropy = mv.s.dual()                                                         # [] Point-
salt = mv.n.dual()                                                            # [] Point-

## The tank

The tank is a strip of triangles one depth high, closed on itself along its length so that it has a floor and a ceiling but no ends. Its vertices are places of unit volume, and an edge that crosses from the last column back to the first reaches a period further along than its ends' places. A corner's hat gradient is the plane through the facing side, along time, volume, entropy and salt, over twice the triangle's area, so that it reads one at the corner. The water starts at rest in two layers, warm and salty over cold and fresh, meeting at a thin interface in the middle whose salinity holds a little noise. Lengths are in depths, times in the time heat takes to conduct across the tank, and temperature and salinity in their differences between the two layers.

In [ ]:
def triangles(topology, edges: Point) -> tuple[Scalar, Point, Plane, Scalar]:
    """Each triangle's area `Scalar+[faces]`, the side facing each corner `[corners] Point-[faces]`, each corner's hat
    gradient `[corners] Plane+[faces]`, and each vertex's share of the area around it `Scalar+[vertices]`."""
    sides = topology.boundary[2, 1] * edges[..., None]                       # [corners] Point-[faces]
    # Twice a triangle's area: two of its sides joined, read through time, at unit volume, entropy and salt.
    double_areas = ((((sides[..., 0] & sides[..., 1]) & time) & volume) & entropy) & salt   # Scalar-[faces]
    areas = double_areas.norm() / 2                                           # Scalar+[faces]
    gradients = (time & volume & entropy & salt & sides) / double_areas[..., None]   # [corners] Plane+[faces]
    return areas, sides, gradients, (~topology.incidence[2, 0] * (areas / 3)).sum(axis=-1)


# The tank, twice as long as it is deep: its topology, its vertices' places, and the seam's shifts.
topology, places, shifts = tank(scenarios.n_along, scenarios.n_across, scenarios.period)
vertices = (volume + axis * places[:, 0] + up * places[:, 1]).field()         # Point-[vertices]
edges = topology.d[0] * vertices + (axis * shifts).field()                    # Point-[edges]
areas, sides, gradients, cells = triangles(topology, edges)
# The two layers at rest, a half above the middle's temperature and salinity at the top and below at the bottom.
start_temperature, start_salinity = scenarios.layered(places, scenarios.disturbance)
temperature = mv.scalar(start_temperature[:, None]).field()                   # Scalar+[vertices]
salinity = mv.scalar(start_salinity[:, None]).field()                         # Scalar+[vertices]
render.draw_tank(topology, vertices, scenarios.n_across, scenarios.period, temperature, salinity);

The tank's temperature, above, red where warm and blue where cold, and its salinity, below, brown where salty and green where fresh: two layers meeting at a thin interface in the middle.

## Fingering

The water's flow is a stream function, whose rise at a corner carries the water along the facing side. Gravity's drive does work on each flow the stream function can make by pulling on the mass the flow carries, `pull & (flows & parcels)`, and since a parcel's mass falls with its warmth and rises with its salt, that one pairing is all the buoyancy of both. The water's mass resists the flow's change and its viscosity the flow's turning. One flux of whole parcels carries their entropy and their salt alike, and each then spreads by its own drive, the heat ten times as fast as the salt, where in seawater it is a hundred, so that the mesh resolves the salt's edges. The density ratio is two: the warmth on top holds up twice the mass its salt adds, so the layers rest stably one on the other until the fingers grow. The frames start just before the fingers show. This cell takes about forty seconds, most of it stepping the water.

In [ ]:
# The water: its density, its viscosity, the Prandtl number in these units, the mass a unit volume loses per degree
# and gains per unit of salinity, salt's diffusivity relative to heat's, and gravity, which with them make the heat's
# Rayleigh number and the density ratio.
density, viscosity, expansion = scenarios.density, scenarios.prandtl, scenarios.expansion
contraction = np.array(expansion / scenarios.ratio)
salt_diffusivity = scenarios.salt_diffusivity
gravity = scenarios.rayleigh * viscosity / expansion


def fingering(topology, areas: Scalar, sides: Point, gradients: Plane, cells: Scalar, contraction: np.ndarray,
              temperature: Scalar, salinity: Scalar, dt: float, n_steps: int,
              stride: int) -> Iterator[tuple[Scalar, Scalar, Scalar]]:
    """A tank's stream function, temperature and salinity `[...] Scalar+[vertices]` every `stride` steps of `dt`,
    from rest and this temperature and salinity, for each contraction of the water by its salt `[...]`, the stream
    function nothing and the temperature and salinity held at the floor and the ceiling."""
    corners = topology.incidence[2, 0]                                        # [corners] Scalar+[faces <- vertices]
    inside = topology.reading(topology.interior[0])                           # Scalar+[free <- vertices]
    # Each corner's share summed back onto its vertex, and the free vertices' values put back among all of them.
    gathering = ~corners                                                      # [corners] Scalar+[vertices <- faces]
    embedding = ~inside                                                       # Scalar+[vertices <- free]
    # The stream function's kinetic form, which is also the conduction of heat, gathered from each triangle's corners
    # onto the vertices they join, and its viscous form.
    local = (gradients[..., :, None] | gradients[..., None, :]) * areas[..., None, None]   # [corners, corners] Scalar+[faces]
    stiffness = (gathering[:, None] * spdiag(local) * corners[None, :]).sum(axis=-1).sum(axis=-1)
    viscous = viscosity * stiffness * spdiag(1 / cells) * stiffness           # Scalar+[vertices <- vertices]
    # Each step's implicit maps on the free vertices: the flow's inertia with its viscosity, the cells' heat with its
    # conduction, and their salt with its slower spreading.
    flowing = (inside * (density / dt * stiffness + viscous) * embedding) * Scalar
    warming = (inside * (spdiag(cells / dt) + stiffness) * embedding) * Scalar
    salting = (inside * (spdiag(cells / dt) + salt_diffusivity * stiffness) * embedding) * Scalar
    # A stream function rising by one at a corner carries half the facing side's worth of water over the triangle.
    flows = sides / 2                                                         # [corners] Point-[faces]
    pull = -(gravity * mv.yt)                                                 # [] Drive+
    # A triangle's share of what its corners hold; the temperature and salinity the walls hold, and the heat and salt
    # they conduct into the free vertices.
    averaging = corners.mean(axis=0)                                          # Scalar+[faces <- vertices]
    held_temperature = temperature - embedding * (inside * temperature)       # [...] Scalar+[vertices]
    held_salinity = salinity - embedding * (inside * salinity)                # [...] Scalar+[vertices]
    wall_heat = inside * (stiffness * held_temperature)                       # [...] Scalar+[free]
    wall_salt = inside * (salt_diffusivity * (stiffness * held_salinity))     # [...] Scalar+[free]
    stream = 0 * temperature                                                  # [...] Scalar+[vertices]
    for step in range(n_steps + 1):
        if step % stride == 0:
            yield stream, temperature, salinity
        # Each triangle's parcels per volume: unit volume, holding less mass the warmer and more the saltier they are,
        # and their entropy and salt.
        warmth = averaging * temperature                                      # [...] Scalar+[faces]
        saltiness = averaging * salinity                                      # [...] Scalar+[faces]
        parcels = (volume + (density - expansion * warmth + contraction * saltiness) * time + warmth * entropy
                   + saltiness * salt)                                        # [...] Point-[faces]
        load = (gathering * (pull & (flows & parcels[..., None]))).sum(axis=-1)   # [...] Scalar+[vertices]
        stream = embedding * flowing.solve(inside * (density / dt * (stiffness * stream) + load))
        # The flow carries the parcels along: what their flux carries out of each corner's cell leaves it, its entropy
        # and its salt alike.
        velocities = ((corners * stream[..., None]) * sides).sum(axis=-1) / (2 * areas)   # [...] Point-[faces]
        flux = velocities & parcels                                           # [...] Flux-[faces]
        carried = (gathering * (areas * (gradients ^ flux[..., None]))).sum(axis=-1)   # [...] Point-[vertices]
        temperature = held_temperature + embedding * warming.solve(
            inside * (cells / dt * temperature - (mv.s & carried)) - wall_heat)
        salinity = held_salinity + embedding * salting.solve(
            inside * (cells / dt * salinity - (mv.n & carried)) - wall_salt)


# Drawn as each frame comes, and kept.
drawn, frames = tee(islice(fingering(topology, areas, sides, gradients, cells, contraction, temperature, salinity,
                                     scenarios.dt, scenarios.n_steps, scenarios.stride), scenarios.n_flat, None))
show_animation(render.animate(topology, vertices, scenarios.n_across, scenarios.period, drawn), "saltfingers",
               scenarios.frame_duration_ms, n_frames=scenarios.n_steps // scenarios.stride + 1 - scenarios.n_flat)

The tank's temperature, above, and its salinity, below, from just before the fingers show. The interface's ripples grow into fingers of salty water sinking and fresh water rising, about ten along the tank, about as wide as heat crosses in the time they take to grow. In the temperature the fingers are blurred, each having traded most of its warmth with its neighbours sideways; in the salinity they keep their sharp edges and nearly all their salt, which is what keeps them moving. Their heads curl into small pairs of vortices, the shape heat–salt fingers take in two dimensions.

## Growth against the density ratio

The more of the salt's mass the warmth holds up, the larger the density ratio and the slower a salty finger sinks once it has lost its warmth; at a density ratio equal to the ratio of the diffusivities, ten here, the heat's head start no longer pays and no disturbance grows. Below a density ratio of one the water is top-heavy and overturns as ordinary convection; the growth rate passes through one without a mark, and the fingers are what still grows above it. The sweep runs water stratified evenly in both heat and salt, held at the floor and the ceiling, at several density ratios at once, a batch of the salt's contractions, on a tank as long as one wavelength of the first rolls to grow between rigid walls; the flow's maps do not depend on the contraction, so every case shares their factorizations. The growth rate is read from the stream function's size at two times. This cell takes about ten seconds.

In [ ]:
# The tank one wavelength of the first rolls long, at these density ratios: the salt's contraction varies, the heat's
# expansion and gravity stay.
ratios = scenarios.growth_ratios                                              # [ratios]
growth_topology, growth_places, growth_shifts = tank(scenarios.growth_along, scenarios.growth_across,
                                                     scenarios.growth_period)
growth_vertices = (volume + axis * growth_places[:, 0] + up * growth_places[:, 1]).field()   # Point-[vertices]
growth_edges = growth_topology.d[0] * growth_vertices + (axis * growth_shifts).field()          # Point-[edges]
growth_areas, growth_sides, growth_gradients, growth_cells = triangles(growth_topology, growth_edges)
# Evenly stratified water, warm and salty above, its salinity disturbed by a roll a wavelength along and half a wave up.
even = mv.scalar((growth_places[:, 1] - 0.5)[:, None]).field()                # Scalar+[vertices]
rolls = (scenarios.growth_disturbance * np.sin(2 * np.pi * growth_places[:, 0] / scenarios.growth_period)
         * np.sin(np.pi * growth_places[:, 1]))
readings = list(fingering(growth_topology, growth_areas, growth_sides, growth_gradients, growth_cells,
                          expansion / ratios, even, even + mv.scalar(rolls[:, None]).field(), scenarios.growth_dt,
                          scenarios.growth_steps, scenarios.growth_stride))
# The stream function's squared size over the tank at the middle and at the end; it grows at twice the rate.
sizes = [(stream * growth_cells * stream).sites.sum() for stream, _, _ in readings[1:]]   # [2] [ratios] Scalar+
growth = (sizes[1].log() - sizes[0].log()) / (2 * scenarios.growth_stride * scenarios.growth_dt)   # [ratios] Scalar+
render.draw_growth(ratios, growth, salt_diffusivity);

The growth rate against the density ratio crosses zero at the dotted line, the ratio of the diffusivities: below it a disturbance grows into fingers, above it the warmth holds the water up and every disturbance dies.

In [ ]:
# checks
# Gravity pulls a parcel's mass down and passes over its volume, its entropy and its salt.
pulled = (mv.y & (-mv.yt).commutator(time + volume + entropy + salt)).to_array()
np.testing.assert_allclose(pulled, -1.0, rtol=1e-12)
# Across the tank's middle the fingers keep their salt and trade their warmth sideways: the salinity there spans nearly
# the whole difference between the layers, and varies more than twice as much as the temperature.
middle = np.isclose(places[:, 1], 0.5)
last = list(frames)[-1]
salinities, temperatures = last[2].to_array()[middle], last[1].to_array()[middle]
assert np.ptp(salinities) > 0.9
assert temperatures.std() < salinities.std() / 2
# The growth rate crosses zero within one percent of the ratio of the diffusivities.
crossing = np.interp(0.0, -growth.to_array(), ratios)
np.testing.assert_allclose(crossing, 1 / salt_diffusivity, rtol=0.01)